# 08 — Hybrid Search: Reciprocal Rank Fusion (RRF)
**Project:** Semantic Book Recommender — IT4142 HUST  
**Owner:** M2c  
**Input:**
- `data/processed/books_with_emotions.csv`
- `models/bm25_index.pkl`
- `data/chroma_db/` (BGE-small embeddings)
- `data/eval/test_queries.json`

**Output:**
- `reports/evaluation_final.json` — updated với Hybrid RRF scores
- `reports/figures/eval_hybrid_analysis.png`

---
## Tại sao Hybrid RRF?

| Model | Điểm mạnh | Điểm yếu |
|---|---|---|
| BM25 | Exact keyword match, nhanh | Fail khi query dùng từ đồng nghĩa |
| Dense (BGE) | Hiểu ngữ nghĩa, paraphrase | Đôi khi bỏ sót từ khóa quan trọng |
| **Hybrid RRF** | **Cả hai** | Chậm hơn BM25 một chút |

**RRF formula:**
$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Với $k=60$ (giá trị chuẩn từ paper gốc Cormack et al. 2009), $R$ = tập các ranked lists.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle, json, time
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_with_emotions.csv')
MODEL_PATH  = Path('models')
CHROMA_PATH = Path('data/chroma_db')
EVAL_PATH   = Path('data/eval/test_queries.json')
REPORT_PATH = Path('reports')
FIGURE_PATH = Path('reports/figures')
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150,
    'figure.facecolor': 'white', 'axes.facecolor': '#F9F9F9',
    'axes.spines.top': False, 'axes.spines.right': False,
})
MODEL_COLORS = {
    'TF-IDF'   : '#6B8CBA',
    'BM25'     : '#F4A261',
    'Semantic' : '#2A9D8F',
    'Hybrid'   : '#E76F51',
    'Reranking': '#8338EC',
}
print('Setup OK')

## 1. Load Data & Models

In [ ]:
df = pd.read_csv(DATA_PATH)
df['isbn13'] = df['isbn13'].astype(str)
print(f'Books: {len(df):,}')

with open(EVAL_PATH) as f:
    test_queries = json.load(f)
print(f'Test queries: {len(test_queries)}')

In [ ]:
# --- BM25 ---
with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)
print(f'BM25 loaded  | corpus: {bm25.corpus_size:,}')

# --- BGE-small + ChromaDB ---
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection('books')
print(f'ChromaDB loaded | count: {collection.count():,}')

BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

## 2. Base Search Functions

In [ ]:
def search_bm25_ids(query: str, top_k: int) -> list[str]:
    """Return ordered list of isbn13 strings."""
    scores = bm25.get_scores(query.lower().split())
    idx = np.argsort(scores)[::-1][:top_k]
    return df.iloc[idx]['isbn13'].tolist()


def search_semantic_ids(query: str, top_k: int) -> list[str]:
    """Return ordered list of isbn13 strings."""
    q_emb = embed_model.encode(
        [BGE_PREFIX + query], normalize_embeddings=True
    )
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['distances'],
    )
    return results['ids'][0]


def get_book_metadata(isbn_list: list[str]) -> pd.DataFrame:
    """Fetch full book rows for a list of isbn13 values, preserving order."""
    isbn_set = set(isbn_list)
    subset = df[df['isbn13'].isin(isbn_set)].copy()
    # Preserve rank order
    order = {isbn: i for i, isbn in enumerate(isbn_list)}
    subset['_order'] = subset['isbn13'].map(order)
    return subset.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print('Base search functions ready')

## 3. Reciprocal Rank Fusion (RRF) Core

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[str]],
    k: int = 60,
    weights: list[float] = None,
) -> list[tuple[str, float]]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists : list of ranked doc-id lists (highest rank = index 0)
        k            : RRF constant. Higher k → less penalty for lower ranks.
                       Standard default = 60 (Cormack et al. 2009)
        weights      : optional per-list weights (default: equal weights)

    Returns:
        List of (doc_id, rrf_score) sorted by descending score
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    assert len(weights) == len(ranked_lists)

    scores: dict[str, float] = defaultdict(float)
    for ranked_list, w in zip(ranked_lists, weights):
        for rank, doc_id in enumerate(ranked_list):
            scores[doc_id] += w * (1.0 / (k + rank + 1))

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def search_hybrid_rrf(
    query: str,
    top_k: int = 10,
    candidate_pool: int = 50,
    k: int = 60,
    weights: list[float] = None,   # [bm25_weight, dense_weight]
) -> pd.DataFrame:
    """
    Hybrid retrieval: BM25 + BGE-small fused with RRF.

    Steps:
      1. BM25 → top-N candidates
      2. BGE-small → top-N candidates
      3. RRF merge → top-K final results
    """
    bm25_ids   = search_bm25_ids(query, top_k=candidate_pool)
    dense_ids  = search_semantic_ids(query, top_k=candidate_pool)

    fused = reciprocal_rank_fusion(
        [bm25_ids, dense_ids],
        k=k,
        weights=weights,
    )

    top_ids = [doc_id for doc_id, _ in fused[:top_k]]
    top_scores = {doc_id: score for doc_id, score in fused[:top_k]}

    result = get_book_metadata(top_ids).copy()
    result['rrf_score'] = result['isbn13'].map(top_scores).round(6)
    return result


# Smoke test
test = search_hybrid_rrf(
    'a heartbreaking story about family secrets in rural America', top_k=5
)
print('Hybrid RRF smoke test:')
test[['title', 'categories', 'rrf_score']]

## 4. RRF Hyperparameter: Effect of k

In [ ]:
def is_relevant(book_category: str, relevant_categories: list) -> bool:
    if not isinstance(book_category, str):
        return False
    bc = book_category.lower()
    return any(rc.lower() in bc or bc in rc.lower() for rc in relevant_categories)

def precision_at_k(results_df: pd.DataFrame, relevant_cats: list, k: int) -> float:
    hits = results_df.head(k)['categories'].apply(
        lambda c: is_relevant(c, relevant_cats)
    ).sum()
    return hits / k

def mrr(results_df: pd.DataFrame, relevant_cats: list, max_k: int = 10) -> float:
    for i, row in results_df.head(max_k).iterrows():
        if is_relevant(row['categories'], relevant_cats):
            return 1.0 / (i + 1)
    return 0.0


# Test different k values on a sample of queries
K_VALUES_RRF = [10, 30, 60, 100]
SAMPLE_QUERIES = test_queries[:20]  # use 20 queries for speed

k_results = []
for rrf_k in K_VALUES_RRF:
    p5_list, mrr_list = [], []
    for q in SAMPLE_QUERIES:
        res = search_hybrid_rrf(q['query'], top_k=10, k=rrf_k)
        p5_list.append(precision_at_k(res, q['relevant_categories'], 5))
        mrr_list.append(mrr(res, q['relevant_categories']))
    k_results.append({'k': rrf_k, 'P@5': np.mean(p5_list), 'MRR': np.mean(mrr_list)})

df_k = pd.DataFrame(k_results)
print('RRF k hyperparameter sensitivity (20 sample queries):')
print(df_k.to_string(index=False))

## 5. Weight Ablation: BM25 vs Dense Contribution

In [ ]:
# Test different weight combinations: [bm25_weight, dense_weight]
WEIGHT_CONFIGS = [
    ([1.0, 0.0], 'BM25 only'),
    ([0.7, 0.3], 'BM25 70% + Dense 30%'),
    ([0.5, 0.5], 'Equal weights'),
    ([0.3, 0.7], 'BM25 30% + Dense 70%'),
    ([0.0, 1.0], 'Dense only'),
]

weight_results = []
for weights, label in WEIGHT_CONFIGS:
    p5_list, mrr_list = [], []
    for q in SAMPLE_QUERIES:
        res = search_hybrid_rrf(q['query'], top_k=10, weights=weights)
        p5_list.append(precision_at_k(res, q['relevant_categories'], 5))
        mrr_list.append(mrr(res, q['relevant_categories']))
    weight_results.append({
        'config'  : label,
        'w_bm25'  : weights[0],
        'w_dense' : weights[1],
        'P@5'     : round(np.mean(p5_list), 4),
        'MRR'     : round(np.mean(mrr_list), 4),
    })

df_weights = pd.DataFrame(weight_results)
print('Weight ablation (20 sample queries):')
print(df_weights.to_string(index=False))

## 6. Full Evaluation — 50 Queries

In [ ]:
print('Running full evaluation (50 queries)...')
t0 = time.time()

hybrid_p5, hybrid_p10, hybrid_mrr = [], [], []
hybrid_records = []

for q in test_queries:
    res = search_hybrid_rrf(q['query'], top_k=10)
    p5  = precision_at_k(res, q['relevant_categories'], 5)
    p10 = precision_at_k(res, q['relevant_categories'], 10)
    m   = mrr(res, q['relevant_categories'])
    hybrid_p5.append(p5)
    hybrid_p10.append(p10)
    hybrid_mrr.append(m)
    hybrid_records.append({
        'query'  : q['query'],
        'genres' : ', '.join(q['relevant_categories']),
        'P@5'    : round(p5, 4),
        'P@10'   : round(p10, 4),
        'MRR'    : round(m, 4),
    })

hybrid_eval = {
    'P@5' : round(np.mean(hybrid_p5),  4),
    'P@10': round(np.mean(hybrid_p10), 4),
    'MRR' : round(np.mean(hybrid_mrr), 4),
}
elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')
print(f'Hybrid RRF results: {hybrid_eval}')

## 7. Plot — Weight Ablation + k Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Plot A: Weight ablation ---
ax = axes[0]
x = np.arange(len(df_weights))
ax.bar(x - 0.2, df_weights['P@5'],  0.35, label='P@5',  color=MODEL_COLORS['Hybrid'],  alpha=0.85)
ax.bar(x + 0.2, df_weights['MRR'],  0.35, label='MRR',  color=MODEL_COLORS['Semantic'], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(
    [f"w=({r['w_bm25']:.1f},{r['w_dense']:.1f})" for _, r in df_weights.iterrows()],
    rotation=30, ha='right', fontsize=8
)
ax.set_title('RRF Weight Ablation\n(BM25 weight, Dense weight)', fontweight='bold')
ax.set_ylabel('Score')
ax.legend()

# --- Plot B: k sensitivity ---
ax2 = axes[1]
ax2.plot(df_k['k'], df_k['P@5'],  marker='o', linewidth=2,
         label='P@5',  color=MODEL_COLORS['Hybrid'])
ax2.plot(df_k['k'], df_k['MRR'],  marker='s', linewidth=2,
         label='MRR',  color=MODEL_COLORS['Semantic'])
ax2.axvline(60, color='grey', linestyle='--', linewidth=1, label='k=60 (default)')
ax2.set_xlabel('RRF k constant')
ax2.set_ylabel('Score')
ax2.set_title('RRF k Hyperparameter Sensitivity', fontweight='bold')
ax2.legend()

plt.suptitle('Hybrid RRF Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_PATH / 'eval_hybrid_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: eval_hybrid_analysis.png')

## 8. Update Evaluation Report

In [ ]:
report_path = REPORT_PATH / 'evaluation_final.json'

if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
else:
    report = {'results': {}}

report['results']['Hybrid'] = {
    **hybrid_eval,
    'type'          : 'dense + sparse',
    'rrf_k'         : 60,
    'candidate_pool': 50,
    'per_query'     : hybrid_records,
}

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Updated: {report_path}')
print(f'Hybrid RRF → P@5={hybrid_eval["P@5"]}  P@10={hybrid_eval["P@10"]}  MRR={hybrid_eval["MRR"]}')

## 9. Qualitative Demo — Vocabulary Mismatch Queries

Đây là bộ queries để **demo trực tiếp** trong báo cáo/slide:  
query dùng ngôn ngữ khác hoàn toàn so với mô tả sách.

In [ ]:
# Load TF-IDF để so sánh
with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sp.load_npz(MODEL_PATH / 'tfidf_matrix.npz')

def search_tfidf(query, top_k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['title', 'categories']].copy()
    res['score'] = scores[idx].round(4)
    return res.reset_index(drop=True)

MISMATCH_QUERIES = [
    'someone losing their mind slowly',           # → "psychological deterioration"
    'a book about a troubled youth finding hope', # → "coming of age", "redemption"
    'far east murder mystery',                    # → "Japan detective", "Tokyo crime"
]

for q in MISMATCH_QUERIES:
    print(f'\n{"="*60}')
    print(f'Query: "{q}"')
    print('\n[TF-IDF]')
    tf = search_tfidf(q, 3)
    print(tf.to_string(index=False))
    print('\n[Hybrid RRF]')
    hy = search_hybrid_rrf(q, top_k=3)[['title', 'categories', 'rrf_score']]
    print(hy.to_string(index=False))

---
## Done ✓

**Key results:**
- `hybrid_eval` dict có P@5, P@10, MRR → dùng trong notebook 07 và báo cáo
- Weight ablation cho thấy configuration tốt nhất
- k=60 là lựa chọn ổn định

**Artifacts:**
- `reports/evaluation_final.json` — updated
- `reports/figures/eval_hybrid_analysis.png`

**Next:** `09_reranking.ipynb` — cross-encoder reranking.